# OT direction, FID at n=50 vs Vacher/Texture Mixer, barycenters

Cell 20: computes `tab:app-vacher`'s FID column -- printed only, no file in the archive holds it.

**Needs:** the rendered midpoints from `literature_baselines_corrected.ipynb`.

**Maintained runnable path:** `experiments/vacher_50.py` reproduces this FID computation from scratch. This notebook is the original source.

In [ ]:
# ============================================================
# Cell 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
results_drive = os.path.join(PROJECT_DIR, "cnt_synthesis")
os.makedirs(results_drive, exist_ok=True)

In [ ]:
# ============================================================
# Cell 2: Install packages
# ============================================================
!pip install -q pot gdown hydra-core omegaconf einops scikit-learn
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# ============================================================
# Cell 3: Clone CNT + fix symlink
# ============================================================
%cd /content
!rm -rf cnt-siga24
!git clone https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24
!pip install -q -r cnt-siga24/requirements.in

ckpts_drive = os.path.join(PROJECT_DIR, "cnt_ckpts")
!rm -rf /content/cnt-siga24/ckpts
!ln -sfn "{ckpts_drive}" /content/cnt-siga24/ckpts
!ls /content/cnt-siga24/ckpts/
%cd /content/cnt-siga24

In [ ]:
# ============================================================
# Cell 4: Load model + imports
# ============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import copy
import random
import sys

sys.path.insert(0, "/content/cnt-siga24")

import torch.nn.functional as F
from torch import nn
from scipy.optimize import linear_sum_assignment
import ot as pot
from sklearn.decomposition import PCA

from src.inference.core import TexAutoEncoderInference, load_image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = TexAutoEncoderInference(
    ckpt_path=Path("/content/cnt-siga24/ckpts/finetuned-model256.ckpt"),
    device=device,
)
print("CNT model loaded!")

In [ ]:
# ============================================================
# Cell 5: Helper functions
# ============================================================
IMG_SIZE = 256

def load_texture(path):
    return load_image(Path(path), IMG_SIZE).to(device)

def get_dtd_pair(cat_a, cat_b, seed=42):
    random.seed(seed)
    img_a = random.choice(os.listdir(os.path.join(dtd_root, cat_a)))
    img_b = random.choice(os.listdir(os.path.join(dtd_root, cat_b)))
    path_a = os.path.join(dtd_root, cat_a, img_a)
    path_b = os.path.join(dtd_root, cat_b, img_b)
    return load_texture(path_a), load_texture(path_b), path_a, path_b

def encode_texture(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor)
        tree_level = blob["layer_tree_levels"][-1][0]
    n_valid = (tree_level["exist_prob"].squeeze(-1) >= 0.5).sum().item()
    print(f"  Valid textons: {n_valid} / {tree_level['xy'].shape[1]}")
    return blob, tree_level

def get_generator_inputs(img_tensor):
    with torch.no_grad():
        blob = model.encode(img_tensor)
        layout = model.splat(blob)
        bf = layout["layer_level_layouts"][-1][0]["blob_features"].clone()
        si = layout["layer_level_layouts"][-1][0]["score_img"].clone()
    return blob, bf, si

def decode_from_blob(blob):
    with torch.no_grad():
        layout = model.splat(blob)
        return model.decode(layout)

def decode_interp(blob_ref, bf_new, si_new):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf_new
        layout["layer_level_layouts"][-1][0]["score_img"] = si_new
        return model.decode(layout)

def build_style_maps(bf, si, feature_map_sizes):
    scores_pyramid = {}
    s = si
    scores_pyramid[s.shape[-1]] = s
    while s.shape[-1] > min(feature_map_sizes):
        s = F.interpolate(s, s.shape[-1] // 2, mode="bilinear", align_corners=False)
        scores_pyramid[s.shape[-1]] = s
    style_maps = {}
    for size in set(feature_map_sizes):
        style_maps[size] = torch.einsum("bnc,bnhw->bchw", bf, scores_pyramid[size])
    return style_maps

def decode_with_style_maps(blob_ref, style_maps_dict):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        si = layout["layer_level_layouts"][-1][0]["score_img"]
        img_size = si.shape[-1]
        ratios = model.generator.feature_map_size_ratios
        feature_map_sizes = [round(img_size * r) for r in ratios]
        scores_pyramid = {}
        s = si
        scores_pyramid[s.shape[-1]] = s
        while s.shape[-1] > min(feature_map_sizes):
            s = F.interpolate(s, s.shape[-1] // 2, mode="bilinear", align_corners=False)
            scores_pyramid[s.shape[-1]] = s
        feature_in = (
            scores_pyramid[feature_map_sizes[0]][:, 1:]
            .amax(dim=1)[:, None]
            .repeat(1, model.generator.n_embedding, 1, 1)
        )
        x = feature_in.clone()
        for i in range(len(feature_map_sizes)):
            style_map = style_maps_dict[feature_map_sizes[i]]
            x = model.generator.spade_blocks[i](x, style_map)
            if i == len(feature_map_sizes) - 1:
                break
            elif feature_map_sizes[i] != feature_map_sizes[i + 1]:
                x = F.interpolate(x, size=x.shape[-1] * 2, mode="bilinear", align_corners=False)
        recon = torch.nn.functional.tanh(x.unsqueeze(1))
        return recon.flatten(1, 2)

def get_feature_map_sizes():
    ratios = model.generator.feature_map_size_ratios
    return [round(IMG_SIZE * r) for r in ratios]

FMS = get_feature_map_sizes()
print(f"Generator feature map sizes: {FMS}")

def tensor_to_image(tensor):
    img = tensor.squeeze(0).cpu().clamp(-1, 1)
    img = (img + 1) / 2
    return img.permute(1, 2, 0).numpy()

In [ ]:
# ============================================================
# Cell 6: Load texture pairs
# ============================================================
texture_pairs = [
    ("striped", "zigzagged", 42),
    ("dotted", "honeycombed", 42),
    ("woven", "knitted", 42),
    ("cracked", "veined", 42),
    ("braided", "spiralled", 42),
    ("scaly", "cobwebbed", 42),
]

pair_data = {}
for cat_a, cat_b, seed in texture_pairs:
    pair_name = f"{cat_a}_vs_{cat_b}"
    print(f"\nLoading {pair_name}...")
    img_a, img_b, pa, pb = get_dtd_pair(cat_a, cat_b, seed=seed)
    blob_a, tree_a = encode_texture(img_a)
    blob_b, tree_b = encode_texture(img_b)
    blob_a_gen, bf_a, si_a = get_generator_inputs(img_a)
    blob_b_gen, bf_b, si_b = get_generator_inputs(img_b)

    pair_data[pair_name] = {
        "img_a": img_a, "img_b": img_b,
        "blob_a": blob_a, "blob_b": blob_b,
        "tree_a": tree_a, "tree_b": tree_b,
        "blob_a_gen": blob_a_gen, "blob_b_gen": blob_b_gen,
        "bf_a": bf_a, "bf_b": bf_b,
        "si_a": si_a, "si_b": si_b,
        "path_a": pa, "path_b": pb,
    }

print(f"\nLoaded {len(pair_data)} pairs!")

In [ ]:
# ============================================================
# Cell 7: Configure experiment + output structure
# ============================================================

import json
import time
from collections import defaultdict
from itertools import combinations
import torchvision.models as models

# Output directory on Drive
EXPERIMENT_DIR = os.path.join(PROJECT_DIR, "interpolation_systematic")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
for sub in ["images", "metrics", "nn_analysis", "summaries"]:
    os.makedirs(os.path.join(EXPERIMENT_DIR, sub), exist_ok=True)

print(f"Experiment output: {EXPERIMENT_DIR}")

# DTD texture categories with structural annotations
TEXTURE_TYPES = {
    "periodic": ["chequered", "grid", "honeycombed", "meshed", "polka-dotted",
                 "striped", "zigzagged", "lined"],
    "regular": ["woven", "knitted", "braided", "interlaced", "pleated",
                "perforated", "lacelike"],
    "non_periodic": ["cracked", "veined", "marbled", "fractured", "crystalline"],
    "no_structure": ["bumpy", "matted", "flecked", "speckled", "blotchy",
                     "smeared", "porous"],
    "strong_local": ["dotted", "scaly", "cobwebbed", "spiralled", "swirly",
                     "paisley", "pitted"],
    "fibrous": ["fibrous", "gauzy", "grooved", "stained", "stratified",
                "wrinkled", "sprinkled"],
}

# Filter to categories that actually exist in DTD
available_categories = sorted([c for c in os.listdir(dtd_root)
                                if os.path.isdir(os.path.join(dtd_root, c))])
print(f"Available DTD categories: {len(available_categories)}")

# Map each category to its type(s)
cat_to_type = {}
for ttype, cats in TEXTURE_TYPES.items():
    for c in cats:
        if c in available_categories:
            cat_to_type[c] = ttype

# Categories without a type get "other"
for c in available_categories:
    if c not in cat_to_type:
        cat_to_type[c] = "other"

print(f"Categorized: {len(cat_to_type)} categories")
for ttype in set(cat_to_type.values()):
    cats = [c for c, t in cat_to_type.items() if t == ttype]
    print(f"  {ttype}: {len(cats)} — {', '.join(cats[:5])}{'...' if len(cats) > 5 else ''}")

In [ ]:
# ============================================================
# Cell 8: VGG perceptual distance + metrics functions
# ============================================================

class VGGPerceptualDistance:
    """Extract VGG16 features at conv1_2, conv2_2, conv3_3, conv4_3
    and compute L2 distance between feature maps."""

    def __init__(self, device='cuda'):
        vgg = models.vgg16(pretrained=True).features.eval().to(device)
        for p in vgg.parameters():
            p.requires_grad = False
        self.device = device

        # Hook into the 4 target layers
        self.features = {}
        target_layers = {3: 'conv1_2', 8: 'conv2_2', 15: 'conv3_3', 22: 'conv4_3'}
        self.slices = []
        prev = 0
        for idx in sorted(target_layers.keys()):
            self.slices.append(torch.nn.Sequential(*list(vgg.children())[prev:idx+1]))
            prev = idx + 1
        for s in self.slices:
            s.eval().to(device)

        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    def extract(self, img):
        """img: (1, 3, 256, 256) in [-1, 1]"""
        x = (img + 1) / 2  # to [0, 1]
        x = (x - self.mean) / self.std
        feats = []
        for s in self.slices:
            x = s(x)
            feats.append(x.clone())
        return feats

    def distance(self, img_a, img_b):
        """Compute perceptual distance between two images."""
        feats_a = self.extract(img_a)
        feats_b = self.extract(img_b)
        d = 0
        for fa, fb in zip(feats_a, feats_b):
            d += (fa - fb).pow(2).sum().item()
        return d

    def distance_batch(self, img_a, images):
        """Distance from img_a to a list of images."""
        feats_a = self.extract(img_a)
        dists = []
        for img in images:
            feats_b = self.extract(img)
            d = sum((fa - fb).pow(2).sum().item() for fa, fb in zip(feats_a, feats_b))
            dists.append(d)
        return dists


vgg_dist = VGGPerceptualDistance(device=device)
print("VGG perceptual distance ready")


def compute_interpolation_metrics(distances, n_steps=21):
    """Given normalized distances at each eta, compute metrics."""
    etas = np.linspace(0, 1, n_steps)

    # Path linearity R²
    from scipy.stats import linregress
    slope, intercept, r_value, _, _ = linregress(etas, distances)
    r_squared = r_value ** 2

    # Midpoint ratio
    mid_idx = n_steps // 2
    midpoint_ratio = distances[mid_idx]

    # Monotonicity (CV of step sizes)
    steps = np.diff(distances)
    if np.mean(np.abs(steps)) > 1e-10:
        monotonicity = np.std(steps) / np.mean(np.abs(steps))
    else:
        monotonicity = 0

    # Symmetry (how similar is forward vs backward)
    forward = distances
    backward = 1 - distances[::-1]
    symmetry_error = np.mean(np.abs(forward - backward))

    return {
        "r_squared": float(r_squared),
        "midpoint_ratio": float(midpoint_ratio),
        "monotonicity": float(monotonicity),
        "symmetry_error": float(symmetry_error),
    }

print("Metrics functions ready")


In [ ]:
# ============================================================
# Cell 9: Sinkhorn OT interpolation implementation
# ============================================================

def sinkhorn_transport(source_feats, target_feats, eps=0.05, n_iters=100):
    """Compute Sinkhorn OT plan between two sets of feature vectors.
    source_feats: (N, D) target_feats: (M, D)
    Returns: transport targets for each source point (N, D)
    """
    N = source_feats.shape[0]
    M = target_feats.shape[0]

    # Cost matrix
    C = torch.cdist(source_feats, target_feats).pow(2)

    # Sinkhorn iterations
    K = torch.exp(-C / eps)
    a = torch.ones(N, 1, device=source_feats.device) / N
    b = torch.ones(M, 1, device=source_feats.device) / M

    u = torch.ones(N, 1, device=source_feats.device)
    for _ in range(n_iters):
        v = b / (K.T @ u + 1e-10)
        u = a / (K @ v + 1e-10)

    # Transport plan
    plan = u * K * v.T  # (N, M)
    # Transport target for each source: weighted average of targets
    plan_normalized = plan / (plan.sum(dim=1, keepdim=True) + 1e-10)
    targets = plan_normalized @ target_feats
    return targets


def interpolate_sinkhorn_pixel(d, eta, n_subsample=4096):
    """Pixel-level Sinkhorn OT interpolation at parameter eta."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    largest = max(FMS)

    source = sm_a[largest][0]  # (C, H, W)
    target = sm_b[largest][0]
    C_ch, H, W = source.shape

    # Flatten to (H*W, C)
    src_flat = source.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = target.permute(1, 2, 0).reshape(-1, C_ch)

    # Subsample
    N = src_flat.shape[0]
    idx_src = torch.randperm(N, device=device)[:n_subsample]
    idx_tgt = torch.randperm(N, device=device)[:n_subsample]

    src_sub = src_flat[idx_src]
    tgt_sub = tgt_flat[idx_tgt]

    # Compute transport targets
    with torch.no_grad():
        transport_targets = sinkhorn_transport(src_sub, tgt_sub, eps=0.05, n_iters=100)

    # Extend to all pixels via nearest-neighbor
    dists_to_sub = torch.cdist(src_flat, src_sub)
    nn_idx = dists_to_sub.argmin(dim=1)
    all_targets = transport_targets[nn_idx]

    # Displacement interpolation
    interp_flat = (1 - eta) * src_flat + eta * all_targets
    interp_sm = interp_flat.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)

    # Build full style_maps dict
    style_maps = {}
    for size in set(FMS):
        if size == largest:
            style_maps[size] = interp_sm
        else:
            style_maps[size] = F.interpolate(interp_sm, size=size,
                                              mode="bilinear", align_corners=False)

    return decode_with_style_maps(d["blob_a_gen"], style_maps)


def interpolate_linear_pixel(d, eta):
    """Pixel-level linear interpolation at parameter eta."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)

    style_maps = {}
    for size in set(FMS):
        style_maps[size] = (1 - eta) * sm_a[size] + eta * sm_b[size]

    return decode_with_style_maps(d["blob_a_gen"], style_maps)


def interpolate_linear_texton(d, eta):
    """Texton-level linear interpolation at parameter eta."""
    bf = (1 - eta) * d["bf_a"] + eta * d["bf_b"]
    si = (1 - eta) * d["si_a"] + eta * d["si_b"]
    return decode_interp(d["blob_a_gen"], bf, si)


print("Interpolation methods ready")

In [ ]:
# ============================================================
# Cell 10: Build ALL texture pairs to test
# ============================================================

print("=" * 70)
print("TASK 1: Building texture pairs")
print("=" * 70)

# Strategy: pick pairs that are
# (a) within the same type (periodic↔periodic, etc.)
# (b) across different types (periodic↔non_periodic, etc.)
# For each category, pick 1 representative image

random.seed(42)
category_images = {}
category_data = {}

for cat in available_categories:
    cat_dir = os.path.join(dtd_root, cat)
    imgs = sorted([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
    if len(imgs) == 0:
        continue
    img_name = random.choice(imgs)
    img_path = os.path.join(cat_dir, img_name)

    try:
        img_tensor = load_texture(img_path)
        blob_gen, bf, si = get_generator_inputs(img_tensor)
        category_images[cat] = img_path
        category_data[cat] = {
            "blob_a_gen": blob_gen, "bf_a": bf, "si_a": si,
            "path_a": img_path,
        }
    except Exception as e:
        print(f"  Skipped {cat}: {e}")

print(f"Loaded {len(category_data)} categories")

# Build pairs: all within-type and a selection of cross-type
all_pairs = []

# Within-type pairs
for ttype, cats in TEXTURE_TYPES.items():
    valid = [c for c in cats if c in category_data]
    for c1, c2 in combinations(valid, 2):
        all_pairs.append((c1, c2, "within", ttype))

# Cross-type pairs (sample to keep manageable)
type_names = list(TEXTURE_TYPES.keys())
for t1, t2 in combinations(type_names, 2):
    valid_t1 = [c for c in TEXTURE_TYPES.get(t1, []) if c in category_data]
    valid_t2 = [c for c in TEXTURE_TYPES.get(t2, []) if c in category_data]
    if not valid_t1 or not valid_t2:
        continue
    # Pick up to 3 cross-type pairs per combination
    random.shuffle(valid_t1)
    random.shuffle(valid_t2)
    for i in range(min(3, len(valid_t1))):
        for j in range(min(1, len(valid_t2))):
            all_pairs.append((valid_t1[i], valid_t2[j], "cross", f"{t1}↔{t2}"))

print(f"Total pairs to test: {len(all_pairs)}")
print(f"  Within-type: {sum(1 for p in all_pairs if p[2] == 'within')}")
print(f"  Cross-type:  {sum(1 for p in all_pairs if p[2] == 'cross')}")


In [ ]:
# ============================================================
# Cell H1: OT + measurement helpers (consolidated from P1/P15/P18/P20)
# ============================================================
from scipy.optimize import linear_sum_assignment

largest = max(FMS)

def exact_ot_targets(X, Y):
    """Exact OT (Hungarian) — un-regularized reference. X, Y: (n, D)."""
    C = torch.cdist(X, Y).pow(2).cpu().numpy()
    r, c = linear_sum_assignment(C)
    T = torch.empty_like(X)
    T[r] = Y[c]
    return T

def rms_amp(F):
    """RMS centered amplitude of a feature cloud (N, D)."""
    return (F - F.mean(0)).pow(2).sum(1).mean().sqrt().item()

def rms_rho(X, Y):
    """Proper (RMS-normalized) correlation under a given pairing; |rho| <= 1."""
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    return (Xc * Yc).sum(1).mean().item() / (rms_amp(X) * rms_amp(Y) + 1e-12)

def w2rel_clouds(P, Q, n=1000):
    """Empirical W2 between clouds / mean cloud spread (Hungarian on subsamples)."""
    Pi = P[torch.randperm(P.shape[0])[:n]]
    Qi = Q[torch.randperm(Q.shape[0])[:n]]
    C = torch.cdist(Pi, Qi).pow(2).cpu().numpy()
    r, c = linear_sum_assignment(C)
    return float(np.sqrt(C[r, c].mean())) / (0.5 * (rms_amp(P) + rms_amp(Q)) + 1e-12)

def reencode(img_tensor):
    """image -> style-map feature cloud (through the encoder)."""
    _, bf, si = get_generator_inputs(img_tensor)
    sm = build_style_maps(bf, si, FMS)[largest][0]
    return sm.permute(1, 2, 0).reshape(-1, sm.shape[0])

print("helpers ready")

In [ ]:
# ============================================================
# Cell H2: pluggable-transport decode (consolidated from P4)
# ============================================================
def decode_with_transport(d, eta, transport_fn, n_subsample=4096):
    """Transport A's style-map features toward B via transport_fn, blend at eta,
    NN-extend to all pixels, decode with A's blob."""
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    source, target = sm_a[largest][0], sm_b[largest][0]
    C_ch, H, W = source.shape
    src_flat = source.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = target.permute(1, 2, 0).reshape(-1, C_ch)

    N = src_flat.shape[0]
    idx_src = torch.randperm(N, device=src_flat.device)[:n_subsample]
    idx_tgt = torch.randperm(N, device=src_flat.device)[:n_subsample]
    src_sub, tgt_sub = src_flat[idx_src], tgt_flat[idx_tgt]

    with torch.no_grad():
        transport_targets = transport_fn(src_sub, tgt_sub)

    nn_idx = torch.cdist(src_flat, src_sub).argmin(dim=1)
    all_targets = transport_targets[nn_idx]

    interp_flat = (1 - eta) * src_flat + eta * all_targets
    interp_sm = interp_flat.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
    style_maps = {}
    for size in set(FMS):
        style_maps[size] = interp_sm if size == largest else F.interpolate(
            interp_sm, size=size, mode="bilinear", align_corners=False)
    return decode_with_style_maps(d["blob_a_gen"], style_maps)

print("decode_with_transport ready")

In [ ]:
# ============================================================
# Cell P25: min-direction vs avg — OT(A->B) vs OT(B->A)
# For each pair: decode BOTH directional midpoints, record empirical
# transport cost per direction, then compare three policies:
#   A-side always | min-cost direction | (avg of the two sides' metrics = OT-sym proxy)
# Metrics: balance, detour (VGG), round-trip realizability.
# NOTE: 'min' implemented as per-pair direction selection by empirical
# Sinkhorn cost <C,P>. If a different min was meant (per-feature, or min
# over a metric), say so and I'll adjust.
# ============================================================
random.seed(303)
_pool25 = [p for p in all_pairs if p[0] in category_data and p[1] in category_data]
random.shuffle(_pool25)
P25_PAIRS = [(a, b) for a, b, *_ in _pool25[:15]]

def sinkhorn_with_cost(X, Y, eps=0.05, n_iters=100):
    C = torch.cdist(X, Y).pow(2)
    K = torch.exp(-C / eps)
    a = torch.ones(X.shape[0], 1, device=X.device) / X.shape[0]
    b = torch.ones(Y.shape[0], 1, device=X.device) / Y.shape[0]
    u = torch.ones_like(a)
    for _ in range(n_iters):
        v = b / (K.T @ u + 1e-10)
        u = a / (K @ v + 1e-10)
    plan = u * K * v.T
    cost = (plan * C).sum().item()
    plan_n = plan / (plan.sum(dim=1, keepdim=True) + 1e-10)
    return plan_n @ Y, cost

def directional_mid(d_src_blob, bf_s, si_s, bf_t, si_t, t):
    """Transport source->target, decode midpoint at t from the source side.
    Returns (image, empirical cost)."""
    sm_s = build_style_maps(bf_s, si_s, FMS)
    sm_t = build_style_maps(bf_t, si_t, FMS)
    src = sm_s[largest][0]; tgt = sm_t[largest][0]
    C_ch, H, W = src.shape
    src_flat = src.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = tgt.permute(1, 2, 0).reshape(-1, C_ch)
    n = 4096
    idx_s = torch.randperm(src_flat.shape[0], device=src_flat.device)[:n]
    idx_t = torch.randperm(tgt_flat.shape[0], device=src_flat.device)[:n]
    with torch.no_grad():
        targets, cost = sinkhorn_with_cost(src_flat[idx_s], tgt_flat[idx_t])
        nn_idx = torch.cdist(src_flat, src_flat[idx_s]).argmin(dim=1)
        all_t = targets[nn_idx]
        interp = ((1 - t) * src_flat + t * all_t).reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
        smaps = {s: interp if s == largest else F.interpolate(
            interp, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
        img = decode_with_style_maps(d_src_blob, smaps)
        rt = w2rel_clouds(reencode(img), (1 - t) * src_flat[idx_s][:1500] + t * targets[:1500])
    return img, cost, rt

print(f"{'pair':30s}{'costA':>8s}{'costB':>8s}{'pick':>5s}"
      f"{'balA':>7s}{'balB':>7s}{'bal_min':>8s}{'bal_avg':>8s}{'det_min':>8s}{'rt_min':>7s}")
p25 = []
for ca, cb in P25_PAIRS:
    ea, eb = category_data[ca], category_data[cb]
    with torch.no_grad():
        img_a = decode_interp(ea["blob_a_gen"], ea["bf_a"], ea["si_a"])
        img_b = decode_interp(eb["blob_a_gen"], eb["bf_a"], eb["si_a"])
        total = vgg_dist.distance(img_a, img_b)
    mA, cA, rtA = directional_mid(ea["blob_a_gen"], ea["bf_a"], ea["si_a"], eb["bf_a"], eb["si_a"], 0.5)
    mB, cB, rtB = directional_mid(eb["blob_a_gen"], eb["bf_a"], eb["si_a"], ea["bf_a"], ea["si_a"], 0.5)
    def bd(img):
        da = vgg_dist.distance(img, img_a); db = vgg_dist.distance(img, img_b)
        return (da - db) / total, (da + db) / total
    balA, detA = bd(mA); balB, detB = bd(mB)
    pick = "A" if cA <= cB else "B"
    bal_min, det_min, rt_min = (balA, detA, rtA) if pick == "A" else (balB, detB, rtB)
    bal_avg = (balA + balB) / 2
    p25.append((f"{ca}_vs_{cb}", cA, cB, pick, balA, balB, bal_min, bal_avg, det_min, rt_min))
    print(f"{ca+'_vs_'+cb:30s}{cA:8.4f}{cB:8.4f}{pick:>5s}"
          f"{balA:+7.2f}{balB:+7.2f}{bal_min:+8.2f}{bal_avg:+8.2f}{det_min:8.2f}{rt_min:7.3f}")

arr25 = lambda i: np.array([r[i] for r in p25])
print(f"\n|balance|: A-side {np.abs(arr25(4)).mean():.3f}   min-pick {np.abs(arr25(6)).mean():.3f}"
      f"   avg-of-sides {np.abs(arr25(7)).mean():.3f}")
print(f"cost asymmetry |cA-cB|/mean: {(np.abs(arr25(1)-arr25(2))/(0.5*(arr25(1)+arr25(2)))).mean():.3f}")
print("reading: min-pick beats A-side on |balance| only if cost asymmetry actually")
print("correlates with the balance sign — check whether 'pick' correlates with the")
print("less-biased side. avg-of-sides is symmetric by construction (P8), so beating")
print("it on |balance| is not the goal; the question is whether min preserves the")
print("single-decode coherence that averaging may lose, at acceptable balance.")

In [ ]:
# ============================================================
# Cell P26: FID of midpoints vs the full DTD
# Methods: reconstruction (the 'penalty' floor), OT sinkhorn e=.05, linear,
# exact OT — plus an optional folder of SOTA outputs (Texture Mixer etc.).
# Fresh random exemplars per pair (not the fixed category_data draws) so the
# midpoint set has real diversity. Folder-based pytorch-fid.
# ============================================================
OUTPUT_ROOT = "/content/drive/MyDrive/thesis_outputs"   # or wherever you want results
os.makedirs(OUTPUT_ROOT, exist_ok=True)
!pip -q install pytorch-fid
from pytorch_fid.fid_score import calculate_fid_given_paths
from PIL import Image as PILImage
import shutil

P26_M = 150            # pairs -> 150 midpoints per method (FID caveat: small-n
                       # FID is biased HIGH; comparisons are valid because n and
                       # pairs are identical across methods — absolute values are not)
P26_REF_CAP = 3000     # reference images from raw DTD
SOTA_DIR = None        # <- set to a folder of SOTA midpoint images to include it

fid_root = os.path.join(OUTPUT_ROOT, "fid_workspace")
dirs = {k: os.path.join(fid_root, k) for k in
        ["ref", "recon", "ot_sinkhorn", "linear", "ot_exact"]}
for p in dirs.values():
    os.makedirs(p, exist_ok=True)

# --- reference: flat copy of raw DTD, capped, spread over categories
ref_count = 0
cats_all = sorted(os.listdir(dtd_root))
per_cat = max(1, P26_REF_CAP // len(cats_all))
for cat in cats_all:
    cdir = os.path.join(dtd_root, cat)
    if not os.path.isdir(cdir): continue
    for f in sorted(os.listdir(cdir))[:per_cat]:
        if f.endswith((".jpg", ".png")):
            shutil.copy(os.path.join(cdir, f), os.path.join(dirs["ref"], f"{cat}_{f}"))
            ref_count += 1
print(f"reference set: {ref_count} raw DTD images")

def save_img(tensor_img, path):
    arr = np.asarray(tensor_to_image(tensor_img))
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    PILImage.fromarray(arr).save(path)

def fresh_pair(seed_i):
    random.seed(9000 + seed_i)
    ca, cb = random.sample(cats_all, 2)
    def draw(cat):
        cdir = os.path.join(dtd_root, cat)
        f = random.choice([x for x in os.listdir(cdir) if x.endswith((".jpg", ".png"))])
        img = load_texture(os.path.join(cdir, f))
        return get_generator_inputs(img)
    return ca, cb, draw(ca), draw(cb)

made, skipped = 0, 0
for i in range(P26_M):
    try:
        ca, cb, (blob_a, bf_a, si_a), (blob_b, bf_b, si_b) = fresh_pair(i)
        d = {"blob_a_gen": blob_a, "bf_a": bf_a, "si_a": si_a, "bf_b": bf_b, "si_b": si_b}
        with torch.no_grad():
            sm_a = build_style_maps(bf_a, si_a, FMS)
            save_img(decode_with_style_maps(blob_a, sm_a),
                     os.path.join(dirs["recon"], f"{i:04d}_{ca}.png"))
            save_img(decode_with_transport(d, 0.5,
                lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)),
                os.path.join(dirs["ot_sinkhorn"], f"{i:04d}.png"))
            save_img(interpolate_linear_pixel(d, 0.5),
                os.path.join(dirs["linear"], f"{i:04d}.png"))
            save_img(decode_with_transport(d, 0.5, exact_ot_targets, n_subsample=1500),
                os.path.join(dirs["ot_exact"], f"{i:04d}.png"))
        made += 1
    except Exception as e:
        skipped += 1
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{P26_M} pairs done ({skipped} skipped)")
print(f"generated {made} midpoints per method")

method_dirs = dict(dirs)
method_dirs.pop("ref")
if SOTA_DIR: method_dirs["sota"] = SOTA_DIR
print(f"\n{'set':14s}{'FID vs DTD':>12s}")
p26 = {}
for name, path in method_dirs.items():
    fid = calculate_fid_given_paths([dirs["ref"], path], batch_size=32,
                                    device="cuda" if torch.cuda.is_available() else "cpu",
                                    dims=2048)
    p26[name] = fid
    print(f"{name:14s}{fid:12.2f}")

print(f"\nreconstruction FID = the 'penalty floor' (best any decoded image can do).")
print(f"Delta over floor: " + "  ".join(
    f"{k}: {p26[k]-p26['recon']:+.1f}" for k in p26 if k != "recon"))
print("claim shape for the paper: OT midpoints sit near the reconstruction floor")
print("(=they 'belong in DTD' up to decoder cost), linear sits materially above.")

In [ ]:
# ============================================================
# Cell P26-fix: resize reference DTD images to a fixed size, rerun FID
# ============================================================
from PIL import Image as PILImage
import shutil

FID_SIZE = 256   # everything FID sees will be this size

# check what size the generated midpoints are, and conform them too if needed
sample = os.listdir(dirs["ot_sinkhorn"])[0]
w, h = PILImage.open(os.path.join(dirs["ot_sinkhorn"], sample)).size
print(f"midpoint image size: {w}x{h}")

# rebuild the reference at fixed size
shutil.rmtree(dirs["ref"]); os.makedirs(dirs["ref"])
ref_count = 0
per_cat = max(1, P26_REF_CAP // len(cats_all))
for cat in cats_all:
    cdir = os.path.join(dtd_root, cat)
    if not os.path.isdir(cdir): continue
    for f in sorted(os.listdir(cdir))[:per_cat]:
        if f.endswith((".jpg", ".png")):
            img = PILImage.open(os.path.join(cdir, f)).convert("RGB")
            # center-crop to square, then resize (avoids aspect distortion)
            s = min(img.size)
            l, t = (img.width - s) // 2, (img.height - s) // 2
            img.crop((l, t, l + s, t + s)).resize(
                (FID_SIZE, FID_SIZE), PILImage.LANCZOS).save(
                os.path.join(dirs["ref"], f"{cat}_{f}".replace(".jpg", ".png")))
            ref_count += 1
print(f"reference rebuilt: {ref_count} images at {FID_SIZE}x{FID_SIZE}")

# conform method images to the same size if they differ
for name, path in method_dirs.items():
    if name == "sota" and SOTA_DIR is None: continue
    for f in os.listdir(path):
        p = os.path.join(path, f)
        img = PILImage.open(p)
        if img.size != (FID_SIZE, FID_SIZE):
            img.convert("RGB").resize((FID_SIZE, FID_SIZE), PILImage.LANCZOS).save(p)

# rerun FID (num_workers=0 sidesteps any residual DataLoader weirdness in Colab)
print(f"\n{'set':14s}{'FID vs DTD':>12s}")
p26 = {}
for name, path in method_dirs.items():
    fid = calculate_fid_given_paths([dirs["ref"], path], batch_size=32,
                                    device="cuda" if torch.cuda.is_available() else "cpu",
                                    dims=2048, num_workers=0)
    p26[name] = fid
    print(f"{name:14s}{fid:12.2f}")

print(f"\nDelta over reconstruction floor: " + "  ".join(
    f"{k}: {p26[k]-p26['recon']:+.1f}" for k in p26 if k != "recon"))

In [ ]:
# ============================================================
# Cell P27: barycenter collapse demo — objects dissolving into one texture
# Encode an image with several distinct objects/regions, cluster its style-map
# features into K groups, compute their (free-support, uniform-weight)
# Wasserstein barycenter, transport every cluster toward it, decode at steps.
# Also logs the K>2 realizability round-trip at each step (v4: untested until now).
# ============================================================
# ============================================================
# Cell P27 (edited): barycenter collapse — COCO bear (285)
# ============================================================
IMG_PATH = os.path.join(coco_dir, "000000000285.jpg")
K_CLUSTERS = 3
P27_TS = [0.0, 0.25, 0.5, 0.75, 1.0]
from sklearn.cluster import KMeans

# square center-crop before encoding (COCO images aren't square)
_pil = PILImage.open(IMG_PATH).convert("RGB")
_s = min(_pil.size)
_l, _t = (_pil.width - _s) // 2, (_pil.height - _s) // 2
_crop_path = "/content/bear_square.png"
_pil.crop((_l, _t, _l + _s, _t + _s)).save(_crop_path)

img = load_texture(_crop_path)
blob, bf, si = get_generator_inputs(img)
sm = build_style_maps(bf, si, FMS)
src = sm[largest][0]
C_ch, H, W = src.shape
Fflat = src.permute(1, 2, 0).reshape(-1, C_ch)

km = KMeans(n_clusters=K_CLUSTERS, n_init=8, random_state=0).fit(Fflat.cpu().numpy())
labels = torch.tensor(km.labels_, device=Fflat.device)

# free-support barycenter (uniform weights, Cuturi–Doucet fixed point, Hungarian steps)
n_bar = 1000
clusters = []
for k in range(K_CLUSTERS):
    Fk = Fflat[labels == k]
    clusters.append(Fk[torch.randperm(Fk.shape[0])[:n_bar]] if Fk.shape[0] >= n_bar
                    else Fk[torch.randint(Fk.shape[0], (n_bar,))])
bar = torch.cat([c[:n_bar // K_CLUSTERS] for c in clusters])[:n_bar].clone()
for it in range(4):
    upd = torch.zeros_like(bar)
    for c in clusters:
        T = exact_ot_targets(bar, c)          # bar -> cluster targets
        upd += T
    bar = upd / K_CLUSTERS
print(f"barycenter fixed-point done ({K_CLUSTERS} clusters, {n_bar} support points)")

# per-cluster transport of ALL pixels toward the barycenter (NN-extended Hungarian)
targets_full = torch.empty_like(Fflat)
for k in range(K_CLUSTERS):
    mask = labels == k
    Tk = exact_ot_targets(clusters[k], bar)    # cluster -> barycenter
    nn_idx = torch.cdist(Fflat[mask], clusters[k]).argmin(dim=1)
    targets_full[mask] = Tk[nn_idx]

fig, axes = plt.subplots(1, len(P27_TS) + 1, figsize=(3.1 * (len(P27_TS) + 1), 3.4))
axes[0].imshow(km.labels_.reshape(H, W), cmap="tab10"); axes[0].axis("off")
axes[0].set_title(f"clusters (K={K_CLUSTERS})", fontsize=9)
between_var, rts = [], []
for j, t in enumerate(P27_TS):
    interp = ((1 - t) * Fflat + t * targets_full)
    # homogenization metric: between-cluster variance of cluster means
    means = torch.stack([interp[labels == k].mean(0) for k in range(K_CLUSTERS)])
    bv = means.var(0).sum().item()
    between_var.append(bv)
    sm_t = interp.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
    smaps = {s: sm_t if s == largest else F.interpolate(
        sm_t, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    with torch.no_grad():
        img_t = decode_with_style_maps(blob, smaps)
        rt = w2rel_clouds(reencode(img_t),
                          interp[torch.randperm(interp.shape[0])[:1500]])
    rts.append(rt)
    axes[j + 1].imshow(tensor_to_image(img_t)); axes[j + 1].axis("off")
    axes[j + 1].set_title(f"t={t}\nrt={rt:.2f}", fontsize=9)
plt.suptitle("Objects dissolving into the Wasserstein barycenter", y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p27_barycenter_collapse.png"), dpi=150,
            bbox_inches="tight")
plt.show()

print(f"between-cluster variance along the path: "
      + " -> ".join(f"{v:.4f}" for v in between_var))
print(f"round-trip along the path:              "
      + " -> ".join(f"{r:.3f}" for r in rts))
print("(the rt sequence IS the K>2 realizability data the v4 status table lists as")
print("untested — if it stays near the ~0.13-0.17 range from P22 at every t, that")
print("row upgrades; a blow-up at intermediate t is itself an informative finding)")

In [ ]:
cat = "marbled"
files = sorted(os.listdir(os.path.join(dtd_root, cat)))[:12]
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for ax, f in zip(axes.flat, files):
    ax.imshow(PILImage.open(os.path.join(dtd_root, cat, f))); ax.axis("off")
    ax.set_title(f, fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
import urllib.request

coco_dir = "/content/coco_samples"
os.makedirs(coco_dir, exist_ok=True)
# a few val2017 images with clear multi-object/multi-texture composition
coco_ids = [39769,   # two cats on a couch (fur / fabric / remote)
            285,     # bear on grass
            632,     # bedroom scene
            724,     # stop sign / street
            776,     # teddy bears
            2149]    # giraffe / trees / sky
for cid in coco_ids:
    url = f"http://images.cocodataset.org/val2017/{cid:012d}.jpg"
    dst = os.path.join(coco_dir, f"{cid:012d}.jpg")
    if not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)
    print(dst)

fig, axes = plt.subplots(1, len(coco_ids), figsize=(3 * len(coco_ids), 3))
for ax, cid in zip(axes, coco_ids):
    ax.imshow(PILImage.open(os.path.join(coco_dir, f"{cid:012d}.jpg"))); ax.axis("off")
    ax.set_title(cid, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell P28: why is exact-OT FID worst? granularity vs smoothing
# Decompose along two axes on the SAME 60 pairs:
#   exact @ n=1500 (P26 setting) | exact @ n=3000 | exact @ n=3000 + local
#   smoothing of the target field | sinkhorn e=.05 (reference)
# If FID drops with n: granularity. If only the smoothed variant closes the
# gap: spatial coherence of the target field is what FID rewards.
# ============================================================
P28_M = 60
smooth_k = 5   # box-blur kernel (in style-map cells) for the smoothed variant

p28_dirs = {k: os.path.join(fid_root, f"p28_{k}") for k in
            ["exact1500", "exact3000", "exact3000_smooth", "sink05"]}
for p in p28_dirs.values():
    shutil.rmtree(p, ignore_errors=True); os.makedirs(p)

def decode_exact_variant(d, eta, n_sub, smooth=False):
    sm_a = build_style_maps(d["bf_a"], d["si_a"], FMS)
    sm_b = build_style_maps(d["bf_b"], d["si_b"], FMS)
    src, tgt = sm_a[largest][0], sm_b[largest][0]
    C_ch, H, W = src.shape
    src_flat = src.permute(1, 2, 0).reshape(-1, C_ch)
    tgt_flat = tgt.permute(1, 2, 0).reshape(-1, C_ch)
    idx_s = torch.randperm(src_flat.shape[0], device=src_flat.device)[:n_sub]
    idx_t = torch.randperm(tgt_flat.shape[0], device=src_flat.device)[:n_sub]
    with torch.no_grad():
        T = exact_ot_targets(src_flat[idx_s], tgt_flat[idx_t])
        nn_idx = torch.cdist(src_flat, src_flat[idx_s]).argmin(dim=1)
        all_t = T[nn_idx]
        if smooth:
            tf = all_t.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
            pad = smooth_k // 2
            tf = F.avg_pool2d(F.pad(tf, (pad,) * 4, mode="reflect"),
                              smooth_k, stride=1)
            all_t = tf[0].permute(1, 2, 0).reshape(-1, C_ch)
        interp = ((1 - eta) * src_flat + eta * all_t
                  ).reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
        smaps = {s: interp if s == largest else F.interpolate(
            interp, size=s, mode="bilinear", align_corners=False)
            for s in set(FMS)}
        return decode_with_style_maps(d["blob_a_gen"], smaps)

made = 0
for i in range(P28_M):
    try:
        ca, cb, (blob_a, bf_a, si_a), (_, bf_b, si_b) = fresh_pair(20000 + i)
        d = {"blob_a_gen": blob_a, "bf_a": bf_a, "si_a": si_a,
             "bf_b": bf_b, "si_b": si_b}
        with torch.no_grad():
            save_img(decode_exact_variant(d, 0.5, 1500),
                     os.path.join(p28_dirs["exact1500"], f"{i:04d}.png"))
            save_img(decode_exact_variant(d, 0.5, 3000),
                     os.path.join(p28_dirs["exact3000"], f"{i:04d}.png"))
            save_img(decode_exact_variant(d, 0.5, 3000, smooth=True),
                     os.path.join(p28_dirs["exact3000_smooth"], f"{i:04d}.png"))
            save_img(decode_with_transport(d, 0.5,
                lambda X, Y: sinkhorn_transport(X, Y, eps=0.05, n_iters=100)),
                os.path.join(p28_dirs["sink05"], f"{i:04d}.png"))
        made += 1
    except Exception:
        pass
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{P28_M} ({made} ok)")

# conform sizes and run FID against the same fixed-size reference
for path in p28_dirs.values():
    for f in os.listdir(path):
        p = os.path.join(path, f)
        im = PILImage.open(p)
        if im.size != (FID_SIZE, FID_SIZE):
            im.convert("RGB").resize((FID_SIZE, FID_SIZE), PILImage.LANCZOS).save(p)

print(f"\n{'variant':20s}{'FID vs DTD':>12s}   (n={made}; compare deltas only,")
print(f"{'':20s}{'':>12s}    and only within this table — different n than P26)")
p28 = {}
for name, path in p28_dirs.items():
    fid = calculate_fid_given_paths([dirs["ref"], path], batch_size=32,
        device="cuda" if torch.cuda.is_available() else "cpu",
        dims=2048, num_workers=0)
    p28[name] = fid
    print(f"{name:20s}{fid:12.2f}")

print("\nreading:")
print("- exact3000 << exact1500  => granularity was the culprit; the P26 gap is an")
print("  artifact of the n=1500 NN extension, soften the 'smoothing helps FID' claim")
print("- exact3000 ~ exact1500 but exact3000_smooth ~ sink05 => spatial coherence of")
print("  the target field is what FID rewards; entropic smoothing earns its keep and")
print("  the epsilon=0.05 defense gets its fourth independent line")
print("- all exact variants above sink05 => smoothing helps through the coupling")
print("  itself (soft plan), not just the field; strongest pro-entropic reading")

In [ ]:
# ============================================================
# Cell P29: clean K>2 realizability — DTD collage barycenter
# 2x2 collage of four in-distribution DTD textures -> encode -> cluster (K=4)
# -> free-support barycenter -> transport all clusters -> decode at steps ->
# round-trip at every t. In-distribution, so rt IS the K>2 datapoint.
# Reference floors: endpoint rt of the collage itself, and P22's ~0.13-0.17.
# ============================================================
COLLAGE_CATS = ["chequered", "woven", "polka-dotted", "grooved"]
P29_TS = [0.0, 0.25, 0.5, 0.75, 1.0]

# build the collage from the category_data exemplars (known-clean encodings)
with torch.no_grad():
    tiles = [tensor_to_image(decode_interp(category_data[c]["blob_a_gen"],
                                           category_data[c]["bf_a"],
                                           category_data[c]["si_a"]))
             for c in COLLAGE_CATS]
tiles = [np.asarray(t) for t in tiles]
ts_ = min(t.shape[0] for t in tiles)
tiles = [t[:ts_, :ts_] for t in tiles]
collage = np.concatenate([np.concatenate(tiles[:2], axis=1),
                          np.concatenate(tiles[2:], axis=1)], axis=0)
collage_path = "/content/collage.png"
PILImage.fromarray(collage if collage.dtype == np.uint8
                   else (np.clip(collage, 0, 1) * 255).astype(np.uint8)).save(collage_path)

img = load_texture(collage_path)
blob, bf, si = get_generator_inputs(img)
sm = build_style_maps(bf, si, FMS)
src = sm[largest][0]
C_ch, H, W = src.shape
Fflat = src.permute(1, 2, 0).reshape(-1, C_ch)

# clusters = the four quadrants BY CONSTRUCTION (no k-means ambiguity)
yy, xx = np.mgrid[0:H, 0:W]
labels = torch.tensor(((yy >= H // 2).astype(int) * 2 + (xx >= W // 2).astype(int)
                       ).reshape(-1), device=Fflat.device)
K = 4

# endpoint floor: round-trip of the collage's own encoding
with torch.no_grad():
    smaps0 = {s: sm[largest] if s == largest else F.interpolate(
        sm[largest], size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    floor_rt = w2rel_clouds(reencode(decode_with_style_maps(blob, smaps0)), Fflat)
print(f"collage endpoint round-trip floor: {floor_rt:.3f}")

n_bar = 1000
clusters = []
for k in range(K):
    Fk = Fflat[labels == k]
    clusters.append(Fk[torch.randperm(Fk.shape[0])[:n_bar]])
bar = torch.cat([c[:n_bar // K] for c in clusters])[:n_bar].clone()
for it in range(4):
    upd = torch.zeros_like(bar)
    for c in clusters:
        upd += exact_ot_targets(bar, c)
    bar = upd / K
print("barycenter fixed-point done")

targets_full = torch.empty_like(Fflat)
for k in range(K):
    mask = labels == k
    Tk = exact_ot_targets(clusters[k], bar)
    nn_idx = torch.cdist(Fflat[mask], clusters[k]).argmin(dim=1)
    targets_full[mask] = Tk[nn_idx]

fig, axes = plt.subplots(1, len(P29_TS) + 1, figsize=(3.1 * (len(P29_TS) + 1), 3.4))
axes[0].imshow(collage); axes[0].axis("off")
axes[0].set_title("input collage", fontsize=9)
bvar, rts = [], []
for j, t in enumerate(P29_TS):
    interp = (1 - t) * Fflat + t * targets_full
    means = torch.stack([interp[labels == k].mean(0) for k in range(K)])
    bvar.append(means.var(0).sum().item())
    sm_t = interp.reshape(H, W, C_ch).permute(2, 0, 1).unsqueeze(0)
    smaps = {s: sm_t if s == largest else F.interpolate(
        sm_t, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    with torch.no_grad():
        img_t = decode_with_style_maps(blob, smaps)
        rt = w2rel_clouds(reencode(img_t),
                          interp[torch.randperm(interp.shape[0])[:1500]])
    rts.append(rt)
    axes[j + 1].imshow(tensor_to_image(img_t)); axes[j + 1].axis("off")
    axes[j + 1].set_title(f"t={t}\nrt={rt:.2f}", fontsize=9)
plt.suptitle("K=4 DTD textures -> Wasserstein barycenter (in-distribution)", y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p29_collage_barycenter.png"), dpi=150,
            bbox_inches="tight")
plt.show()

print(f"between-cluster variance: " + " -> ".join(f"{v:.4f}" for v in bvar))
print(f"round-trip:               " + " -> ".join(f"{r:.3f}" for r in rts))
print(f"floors: collage endpoint {floor_rt:.3f}; P22 pairwise range ~0.13-0.17")
print("\nreading: rt staying within ~1.5x the collage floor at ALL t => K>2 geodesic")
print("marginals are realizable and the v4 status row upgrades to 'supported (K=4)'.")
print("rt rising with t toward the full barycenter => the barycenter of 4 distant")
print("textures sits outside the realizable set even though pairwise midpoints don't —")
print("a real boundary finding, and directly relevant to the mixture experiments.")

In [ ]:
import os
# ============================================================
# Cell P30: SOTA FID column — Texture Mixer + Vacher vs ours, fair protocol
# All sets at 128px (TM's native res = the paper's parity point), all
# subsampled to a common n, against a 128px DTD reference.
# Also extracts per-method timing medians from the Vacher manifest
# (the speed-accounting data for the 215x claim).
# ============================================================
import glob as _glob

BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_MID = f'{BASE}/texturemixer/midpoints'

# auto-locate the Vacher lit_midpoints dir
_hits = _glob.glob(f'{BASE}/*/lit_midpoints') + _glob.glob(f'{BASE}/lit_midpoints')
assert _hits, "lit_midpoints not found under BASE — set VAC_MID manually"
VAC_MID = _hits[0]
VAC_MANIFEST = os.path.join(os.path.dirname(VAC_MID), 'lit_manifest.csv')
print("TM: ", TM_MID, "\nVAC:", VAC_MID)

FID128 = 128
p30_root = os.path.join(fid_root, "p30_128")
sets128 = {}

# --- gather each method's file list
import pandas as pd
md = pd.read_csv(VAC_MANIFEST); md = md[md.ok]
vac_methods = ["tex_wasser", "tex_gram", "tex_gauss"]
gather = {
    "texture_mixer": sorted(_glob.glob(f"{TM_MID}/*_mid.png")),
}
for m in vac_methods:
    gather[m] = sorted(md[md.method == m].path.tolist())
# ours: reuse P26's generated sets
for m in ["recon", "ot_sinkhorn", "linear"]:
    gather[m] = sorted(_glob.glob(os.path.join(dirs[m], "*.png")))

n_common = min(len(v) for v in gather.values())
print(f"\nset sizes: " + ", ".join(f"{k}:{len(v)}" for k, v in gather.items()))
print(f"common n = {n_common}  (all sets subsampled to this; FID comparable within table only)")

# --- write everything at 128px
rng = random.Random(55)
for name, files in gather.items():
    outd = os.path.join(p30_root, name)
    shutil.rmtree(outd, ignore_errors=True); os.makedirs(outd)
    for i, f in enumerate(rng.sample(files, n_common)):
        try:
            PILImage.open(f).convert("RGB").resize(
                (FID128, FID128), PILImage.LANCZOS).save(os.path.join(outd, f"{i:04d}.png"))
        except Exception as e:
            print(f"  skip {f}: {e}")
    sets128[name] = outd

# --- 128px reference from raw DTD (center-crop square, then resize)
ref128 = os.path.join(p30_root, "ref128")
shutil.rmtree(ref128, ignore_errors=True); os.makedirs(ref128)
per_cat = max(1, P26_REF_CAP // len(cats_all))
rc = 0
for cat in cats_all:
    cdir = os.path.join(dtd_root, cat)
    if not os.path.isdir(cdir): continue
    for f in sorted(os.listdir(cdir))[:per_cat]:
        if f.endswith((".jpg", ".png")):
            img = PILImage.open(os.path.join(cdir, f)).convert("RGB")
            s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
            img.crop((l, t, l + s, t + s)).resize((FID128, FID128), PILImage.LANCZOS
                ).save(os.path.join(ref128, f"{cat}_{f}".replace(".jpg", ".png")))
            rc += 1
print(f"reference: {rc} images at {FID128}px")

# --- FID table
print(f"\n{'set':16s}{'FID vs DTD (128px)':>20s}")
p30 = {}
order = ["recon", "ot_sinkhorn", "linear", "texture_mixer",
         "tex_wasser", "tex_gram", "tex_gauss"]
for name in order:
    fid = calculate_fid_given_paths([ref128, sets128[name]], batch_size=32,
        device="cuda" if torch.cuda.is_available() else "cpu",
        dims=2048, num_workers=0)
    p30[name] = fid
    print(f"{name:16s}{fid:20.2f}")

print(f"\nDelta over reconstruction floor:")
for k in order:
    if k != "recon":
        print(f"  {k:16s}{p30[k] - p30['recon']:+8.1f}")

# --- timing accounting from the Vacher manifest
print(f"\n{'method':16s}{'median secs/midpoint':>22s}{'n':>6s}")
for m in sorted(md.method.unique()):
    sub = md[md.method == m]
    print(f"{m:16s}{sub.secs.median():22.2f}{len(sub):6d}")
print("(compare against your ~0.11s sinkhorn e=.05 from P9; TM's median was printed")
print(" in the TF notebook's generation cell — pull that number for the same table)")

In [ ]:
# recon and ot_sinkhorn at full n=139, 128px, same reference
for name in ["recon", "ot_sinkhorn"]:
    outd = os.path.join(p30_root, name + "_full")
    shutil.rmtree(outd, ignore_errors=True); os.makedirs(outd)
    for i, f in enumerate(sorted(_glob.glob(os.path.join(dirs[name], "*.png")))):
        PILImage.open(f).convert("RGB").resize((FID128, FID128), PILImage.LANCZOS
            ).save(os.path.join(outd, f"{i:04d}.png"))
    fid = calculate_fid_given_paths([ref128, outd], batch_size=32,
        device="cuda" if torch.cuda.is_available() else "cpu", dims=2048, num_workers=0)
    print(f"{name}_full (n=139): {fid:.2f}")

In [ ]:
# ============================================================
# Cell P31: pool — encode ~8/category, fixed-support clouds, rho* screening
# ============================================================
import math
SEED_B = 500
N_PER_CAT = 8
SUPPORT_N = 2048

pool_paths = []
for c in cats_all:
    cdir = os.path.join(dtd_root, c)
    if not os.path.isdir(cdir): continue
    fs = sorted(_glob.glob(os.path.join(cdir, "*.jpg")))
    pool_paths += random.Random(f"{SEED_B}-{c}").sample(fs, min(N_PER_CAT, len(fs)))
print(f"pool: {len(pool_paths)} images")

POOL, ok_paths, t0 = [], [], time.time()
for i, p in enumerate(pool_paths):
    try:
        img = load_texture(p)
        _, bf, si = get_generator_inputs(img)
        sm = build_style_maps(bf, si, FMS)[largest][0]
        F_ = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
        idx = torch.randperm(F_.shape[0], device=F_.device)[:SUPPORT_N]
        POOL.append(F_[idx].cpu())
        ok_paths.append(p)
    except Exception:
        pass
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(pool_paths)} ({(time.time()-t0)/60:.1f}m)")
pool_paths = ok_paths
N_POOL = len(POOL)
C_DIM = POOL[0].shape[1]
cat_of = lambda p: os.path.basename(os.path.dirname(p))
print(f"encoded {N_POOL} clouds, {SUPPORT_N} x {C_DIM}")

# rho*-style screen: mean-offset from the pool's global center, in spread units
# (the cheap sufficient statistic — all three past catches were 7-8 units out)
global_mean = torch.stack([P.mean(0) for P in POOL]).mean(0)
spreads = np.array([rms_amp(P) for P in POOL])
offsets = np.array([(P.mean(0) - global_mean).norm().item() for P in POOL]) / spreads.mean()
flag_mask = offsets > 4.0
print(f"\nscreen: {int(flag_mask.sum())} flagged encodings (mean-offset > 4 spread units):")
for i in np.where(flag_mask)[0]:
    print(f"  {cat_of(pool_paths[i])}/{os.path.basename(pool_paths[i])}  offset={offsets[i]:.1f}")
CLEAN = np.where(~flag_mask)[0].tolist()
print(f"clean pool: {len(CLEAN)}  (basis selection and landmarks use CLEAN only;")
print(" flagged items stay as held-out curiosities — do NOT delete, they're catches)")

In [ ]:
# ============================================================
# Cell P32: landmark distance matrix (proposed)
# L=50 landmarks -> L x N sliced-W2 distance matrix (fast, near-linear),
# validated against entropic Sinkhorn on random entries so the metric
# choice is a measured approximation, not an assumption.
# ============================================================
L_LANDMARKS = 50
N_PROJ = 128
dev = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED_B)
PROJ = torch.randn(C_DIM, N_PROJ, device=dev)
PROJ = PROJ / PROJ.norm(dim=0, keepdim=True)

@torch.no_grad()
def sw_signature(P):
    """sorted projections: (SUPPORT_N, N_PROJ) — sliced-W2 becomes L2 here,
    and sliced-W barycenters become weighted averages of signatures."""
    return torch.sort((P.to(dev) @ PROJ), dim=0).values

print("building signatures...")
SIGS = torch.stack([sw_signature(P) for P in POOL])          # (N, S, P)
SIGS_flat = SIGS.reshape(N_POOL, -1)

def sw2(i, j):
    return float((SIGS_flat[i] - SIGS_flat[j]).pow(2).mean().sqrt())

# landmarks: k-medoids++ style greedy on the clean pool (spread but not maximal)
rng = random.Random(SEED_B)
landmarks = [rng.choice(CLEAN)]
while len(landmarks) < L_LANDMARKS:
    d2 = torch.stack([ (SIGS_flat[CLEAN] - SIGS_flat[l]).pow(2).mean(1)
                       for l in landmarks]).min(0).values
    probs = (d2 / d2.sum()).cpu().numpy()          # ++ sampling, not argmax (anti-outlier)
    landmarks.append(CLEAN[np.random.default_rng(len(landmarks)).choice(len(CLEAN), p=probs)])
landmarks = sorted(set(landmarks))
print(f"{len(landmarks)} landmarks from {len({cat_of(pool_paths[l]) for l in landmarks})} categories")

# L x N distance matrix (this is the proposed matrix)
D_LN = torch.cdist(SIGS_flat[landmarks], SIGS_flat) / math.sqrt(SIGS_flat.shape[1])
print(f"distance matrix: {tuple(D_LN.shape)}")

# validate sliced-W against entropic Sinkhorn on 60 random entries
@torch.no_grad()
def sink_dist(i, j, eps=0.05, iters=100):
    X, Y = POOL[i][:1200].to(dev), POOL[j][:1200].to(dev)
    C = torch.cdist(X, Y).pow(2)
    K = torch.exp(-C / eps)
    a = torch.ones(X.shape[0], 1, device=dev) / X.shape[0]
    b = torch.ones(Y.shape[0], 1, device=dev) / Y.shape[0]
    u = torch.ones_like(a)
    for _ in range(iters):
        v = b / (K.T @ u + 1e-10); u = a / (K @ v + 1e-10)
    return float(((u * K * v.T) * C).sum().sqrt())

from scipy.stats import spearmanr
val = [(rng.choice(landmarks), rng.choice(CLEAN)) for _ in range(60)]
sw_v  = [sw2(i, j) for i, j in val]
snk_v = [sink_dist(i, j) for i, j in val]
sp = spearmanr(sw_v, snk_v)
print(f"sliced-W vs Sinkhorn-W: spearman r={sp.statistic:.3f} (p={sp.pvalue:.1e})")
print("(r > ~0.9 => the fast metric preserves the geometry and the matrix is trustworthy;")
print(" lower => rerun D_LN with sink_dist — 50x350 x ~0.1s ~ 30-40 min, still feasible)")

In [ ]:
# ============================================================
# Cell P33: Nystrom / Landmark-MDS — complete the geometry from L x N
# Embed all N pool textures into R^k from landmark distances only.
# ============================================================
D_LL = D_LN[:, landmarks]                                   # (L, L)
D2 = (D_LL ** 2).cpu().numpy()
L = len(landmarks)
J = np.eye(L) - np.ones((L, L)) / L
B = -0.5 * J @ D2 @ J                                       # double-centered Gram
evals, evecs = np.linalg.eigh(B)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
k_emb = int(np.searchsorted(np.cumsum(np.clip(evals, 0, None)) /
                            np.clip(evals, 0, None).sum(), 0.95)) + 1
k_emb = max(k_emb, 8)
print(f"embedding dim k={k_emb} (95% of landmark spectrum)")

Lam = np.clip(evals[:k_emb], 1e-10, None)
Lk = evecs[:, :k_emb] * np.sqrt(Lam)                        # landmark coords (L, k)
# out-of-sample (Nystrom) projection for every pool item
mean_d2_l = D2.mean(axis=1)
D2_all = (D_LN ** 2).cpu().numpy()                          # (L, N)
Z = ((mean_d2_l[:, None] - D2_all).T @ (evecs[:, :k_emb] / np.sqrt(Lam))) / 2   # (N, k)
print(f"embedded all {N_POOL} textures into R^{k_emb}")

# sanity: same-category should be closer than cross-category in the embedding
from itertools import combinations as _comb
cats_arr = np.array([cat_of(p) for p in pool_paths])
d_within, d_cross = [], []
for _ in range(3000):
    i, j = rng.randrange(N_POOL), rng.randrange(N_POOL)
    if i == j: continue
    d = np.linalg.norm(Z[i] - Z[j])
    (d_within if cats_arr[i] == cats_arr[j] else d_cross).append(d)
print(f"embedding sanity: within-cat dist {np.mean(d_within):.3f}  "
      f"cross-cat {np.mean(d_cross):.3f}  (within < cross expected)")

In [ ]:
# ============================================================
# Cell P34: basis selection on the completed matrix — head-to-head
# random | FPS(embed) | k-medoids(embed) | leverage/CUR | max-volume |
# archetypal analysis. Evaluated by simplex barycentric coding of held-out
# targets in signature space (where sliced-W barycenters ARE linear mixes).
# ============================================================
from sklearn.cluster import KMeans
M_BASIS = 40          # basis size; sweep later if one method wins clearly
K_SPARSE = 3          # atoms per code (matches your earlier protocol)
N_TARGETS = 40

Zc = Z[CLEAN]; clean_arr = np.array(CLEAN)

def sel_random(m):  return clean_arr[np.random.default_rng(1).choice(len(CLEAN), m, replace=False)]
def sel_fps(m):
    pts = [int(np.argmin(np.linalg.norm(Zc - Zc.mean(0), axis=1)))]
    for _ in range(m - 1):
        d = np.min(np.linalg.norm(Zc[:, None] - Zc[pts][None], axis=2), axis=1)
        pts.append(int(np.argmax(d)))
    return clean_arr[pts]
def sel_kmed(m):
    km = KMeans(m, n_init=8, random_state=0).fit(Zc)
    return clean_arr[[int(np.argmin(np.linalg.norm(Zc - c, axis=1)))
                      for c in km.cluster_centers_]]
def sel_leverage(m):
    U, S, _ = np.linalg.svd(Zc - Zc.mean(0), full_matrices=False)
    lev = (U[:, :k_emb] ** 2).sum(1)
    return clean_arr[np.argsort(-lev)[:m]]
def sel_maxvol(m):
    X = Zc - Zc.mean(0); pts = [int(np.argmax(np.linalg.norm(X, axis=1)))]
    for _ in range(m - 1):
        Q, _ = np.linalg.qr(X[pts].T)
        resid = X - (X @ Q) @ Q.T
        pts.append(int(np.argmax(np.linalg.norm(resid, axis=1))))
    return clean_arr[pts]
def sel_archetypes(m, iters=60):
    """PCHA-lite: archetypes = convex combos of data, data ≈ convex combos of
    archetypes — THE dictionary objective matched to simplex coding."""
    X = torch.tensor(Zc, dtype=torch.float32, device=dev)
    n = X.shape[0]
    Cm = torch.rand(n, m, device=dev); Cm /= Cm.sum(0, keepdim=True)
    Sm = torch.rand(m, n, device=dev); Sm /= Sm.sum(0, keepdim=True)
    for _ in range(iters):
        A = Cm.T @ X                                   # archetypes (m, k)
        # S-step: project each column's gradient step to simplex
        G = A @ A.T @ Sm - A @ X.T
        Sm = torch.softmax(torch.log(Sm.clamp_min(1e-12)) - 0.5 * G, dim=0)
        # C-step
        R = X.T @ (Sm.T @ torch.linalg.pinv(A @ A.T + 1e-6 * torch.eye(m, device=dev)))
        Cm = torch.softmax(torch.log(Cm.clamp_min(1e-12)) + 0.0 * Cm - 0.5 *
                           (X @ X.T @ Cm - X @ R).clamp(-5, 5), dim=0)
    A = (Cm.T @ X).cpu().numpy()
    return clean_arr[[int(np.argmin(np.linalg.norm(Zc - a, axis=1))) for a in A]]

SELECTORS = {"random": sel_random, "fps_embed": sel_fps, "kmedoids": sel_kmed,
             "leverage": sel_leverage, "maxvol": sel_maxvol,
             "archetypes": sel_archetypes}

# held-out targets (fresh images, not in pool)
targets = []
seen = set(pool_paths)
while len(targets) < N_TARGETS:
    c = rng.choice(cats_all)
    fs = _glob.glob(os.path.join(dtd_root, c, "*.jpg"))
    if not fs: continue
    p = rng.choice(fs)
    if p in seen: continue
    try:
        img = load_texture(p); _, bf, si = get_generator_inputs(img)
        sm = build_style_maps(bf, si, FMS)[largest][0]
        F_ = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
        targets.append(sw_signature(F_[torch.randperm(F_.shape[0])[:SUPPORT_N]]).reshape(-1))
        seen.add(p)
    except Exception:
        continue
targets = torch.stack(targets)
print(f"{len(targets)} held-out targets")

def simplex_project(v):
    u, _ = torch.sort(v, descending=True); css = torch.cumsum(u, 0)
    kk = torch.arange(1, len(v) + 1, device=v.device, dtype=v.dtype)
    r = int((u - (css - 1) / kk > 0).nonzero()[-1]) + 1
    return (v - (css[r - 1] - 1) / r).clamp_min(0)

@torch.no_grad()
def code_residual(t_sig, basis_idx):
    """k-sparse simplex coding in signature space: shortlist by distance,
    fit simplex weights on best K_SPARSE subset."""
    Bsig = SIGS_flat[torch.tensor(basis_idx)]
    d = (Bsig - t_sig[None]).pow(2).mean(1)
    short = torch.argsort(d)[:8]
    best = None
    from itertools import combinations
    for combo in combinations(short.tolist(), K_SPARSE):
        Bm = Bsig[list(combo)]
        G = Bm @ Bm.T; b = Bm @ t_sig
        lr = 1.0 / max(float(torch.linalg.eigvalsh(G.double()).max()), 1e-12)
        w = torch.full((K_SPARSE,), 1 / K_SPARSE, device=Bm.device)
        for _ in range(120):
            w = simplex_project(w - lr * (G @ w - b))
        r = float((Bm.T @ w - t_sig).norm() / t_sig.norm())
        if best is None or r < best:
            best = r
    return best

print(f"\n{'selector':12s}{'median resid':>13s}{'mean resid':>12s}{'cats in basis':>14s}")
p34 = {}
for name, fn in SELECTORS.items():
    basis = fn(M_BASIS)
    res = [code_residual(t, basis) for t in targets]
    p34[name] = res
    ncat = len({cat_of(pool_paths[i]) for i in basis})
    print(f"{name:12s}{np.median(res):13.4f}{np.mean(res):12.4f}{ncat:14d}")

from scipy.stats import wilcoxon
base = p34["kmedoids"]
print("\nvs k-medoids (your previous champion), Wilcoxon:")
for name, res in p34.items():
    if name == "kmedoids": continue
    w = wilcoxon(res, base)
    better = "better" if np.median(res) < np.median(base) else "worse"
    print(f"  {name:12s} {better:7s} p={w.pvalue:.3f}")

In [ ]:
# ============================================================
# Cell P35: attribution ablation — screen vs embedding, 2x2
# FPS run: {embedding, raw signature space} x {screened, unscreened}
# ============================================================
def sel_fps_generic(m, pool_idx, coords):
    pts = [int(np.argmin(np.linalg.norm(coords - coords.mean(0), axis=1)))]
    for _ in range(m - 1):
        d = np.min(np.linalg.norm(coords[:, None] - coords[pts][None], axis=2), axis=1)
        pts.append(int(np.argmax(d)))
    return np.array(pool_idx)[pts]

ALL_IDX = list(range(N_POOL))
SIG_np = SIGS_flat.cpu().numpy()

variants = {
    "embed+screen":   sel_fps_generic(M_BASIS, CLEAN,  Z[CLEAN]),
    "embed+all":      sel_fps_generic(M_BASIS, ALL_IDX, Z),
    "rawsig+screen":  sel_fps_generic(M_BASIS, CLEAN,  SIG_np[CLEAN]),
    "rawsig+all":     sel_fps_generic(M_BASIS, ALL_IDX, SIG_np),
}
print(f"{'variant':>15s}{'median resid':>13s}{'flagged in basis':>17s}")
for name, basis in variants.items():
    res = [code_residual(t, basis) for t in targets]
    nflag = int(sum(1 for i in basis if i not in CLEAN))
    print(f"{name:>15s}{np.median(res):13.4f}{nflag:17d}")
print("\nreading: screen variants << all variants (esp. rawsig) => the screen is the")
print("fix and the old FPS failure is fully explained. embed ~ rawsig => the Nystrom")
print("step is a convenience, not the cause; embed << rawsig => the completed matrix")
print("itself denoises the geometry — credit the proposed pipeline directly.")

In [ ]:
# ============================================================
# Cell P36: robustness — basis-size sweep for the two contenders
# ============================================================
print(f"{'M':>5s}{'fps_embed':>12s}{'kmedoids':>12s}{'p (Wilcoxon)':>14s}")
for M in [20, 40, 60, 80]:
    r_f = [code_residual(t, sel_fps_generic(M, CLEAN, Z[CLEAN])) for t in targets]
    km = KMeans(M, n_init=8, random_state=0).fit(Z[CLEAN])
    b_k = np.array(CLEAN)[[int(np.argmin(np.linalg.norm(Z[CLEAN] - c, axis=1)))
                            for c in km.cluster_centers_]]
    r_k = [code_residual(t, b_k) for t in targets]
    w = wilcoxon(r_f, r_k)
    print(f"{M:5d}{np.median(r_f):12.4f}{np.median(r_k):12.4f}{w.pvalue:14.3f}")
print("\nif FPS's edge holds (or grows) across M and especially at M=80 — your old")
print("champion's operating point — the section's conclusion is settled; if it only")
print("wins at M=40, report it as size-dependent and keep both in the table.")

In [ ]:
# ============================================================
# Cell P37: hero figures — original | CNT recon | coded (FPS-40) | coded (kmed-80)
# ============================================================
N_FIG = 6
BASIS_FPS  = sel_fps_generic(40, CLEAN, Z[CLEAN])
km80 = KMeans(80, n_init=8, random_state=0).fit(Z[CLEAN])
BASIS_KMED = np.array(CLEAN)[[int(np.argmin(np.linalg.norm(Z[CLEAN] - c, axis=1)))
                              for c in km80.cluster_centers_]]

# fresh held-out targets WITH full data this time
fig_targets = []
seen = set(pool_paths)
while len(fig_targets) < N_FIG:
    c = rng.choice(cats_all)
    fs = _glob.glob(os.path.join(dtd_root, c, "*.jpg"))
    if not fs: continue
    p = rng.choice(fs)
    if p in seen: continue
    try:
        img = load_texture(p)
        blob, bf, si = get_generator_inputs(img)
        sm = build_style_maps(bf, si, FMS)[largest][0]
        Ff = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
        X = Ff[torch.randperm(Ff.shape[0])[:SUPPORT_N]]
        fig_targets.append(dict(path=p, blob=blob, bf=bf, si=si, Ff=Ff, X=X,
                                sig=sw_signature(X).reshape(-1)))
        seen.add(p)
    except Exception:
        continue
print(f"{N_FIG} figure targets encoded")

from itertools import combinations
@torch.no_grad()
def code_with_atoms(t_sig, basis_idx):
    Bsig = SIGS_flat[torch.tensor(basis_idx)]
    d = (Bsig - t_sig[None]).pow(2).mean(1)
    short = torch.argsort(d)[:8]
    best = (None, None, np.inf)
    for combo in combinations(short.tolist(), K_SPARSE):
        Bm = Bsig[list(combo)]
        G = Bm @ Bm.T; b = Bm @ t_sig
        lr = 1.0 / max(float(torch.linalg.eigvalsh(G.double()).max()), 1e-12)
        w = torch.full((K_SPARSE,), 1 / K_SPARSE, device=Bm.device)
        for _ in range(120):
            w = simplex_project(w - lr * (G @ w - b))
        r = float((Bm.T @ w - t_sig).norm() / t_sig.norm())
        if r < best[2]:
            best = ([int(basis_idx[i]) for i in combo], w.clone(), r)
    return best

@torch.no_grad()
def hard_map(X_t, atom_cloud, eps_rel=0.04, iters=80):
    """argmax transport: each target point takes a REAL atom feature."""
    Y = atom_cloud.to(X_t.device)
    C = torch.cdist(X_t, Y).pow(2)
    logK = -(C / C.median().clamp_min(1e-8)) / eps_rel
    n, m = C.shape
    la = torch.full((n, 1), -math.log(n), device=X_t.device)
    lb = torch.full((m, 1), -math.log(m), device=X_t.device)
    lu, lv = torch.zeros_like(la), torch.zeros_like(lb)
    for _ in range(iters):
        lu = la - torch.logsumexp(logK + lv.T, dim=1, keepdim=True)
        lv = lb - torch.logsumexp(logK.T + lu.T, dim=1, keepdim=True)
    return Y[(logK + lv.T).argmax(dim=1)]

@torch.no_grad()
def decode_coded(t, atoms, w):
    mix = None
    for wi, j in zip(w, atoms):
        if float(wi) < 1e-4: continue
        mapped = hard_map(t["X"], POOL[j])
        mix = wi * mapped if mix is None else mix + wi * mapped
    nn = torch.cdist(t["Ff"], t["X"]).argmin(dim=1)
    full = mix[nn]
    C_ = full.shape[1]
    Hs = Ws = int(math.isqrt(t["Ff"].shape[0]))
    sm_t = full.reshape(Hs, Ws, C_).permute(2, 0, 1).unsqueeze(0)
    smaps = {s: sm_t if s == largest else F.interpolate(
        sm_t, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    return decode_with_style_maps(t["blob"], smaps)

cols = ["original", "CNT recon", "coded FPS-40", "coded kmed-80"]
fig, axes = plt.subplots(N_FIG, 4, figsize=(12.5, 3.1 * N_FIG))
for r, t in enumerate(fig_targets):
    with torch.no_grad():
        sm0 = build_style_maps(t["bf"], t["si"], FMS)
        recon = decode_with_style_maps(t["blob"], sm0)
    aF, wF, rF = code_with_atoms(t["sig"], BASIS_FPS)
    aK, wK, rK = code_with_atoms(t["sig"], BASIS_KMED)
    imgs = [np.asarray(PILImage.open(t["path"]).convert("RGB")),
            tensor_to_image(recon),
            tensor_to_image(decode_coded(t, aF, wF)),
            tensor_to_image(decode_coded(t, aK, wK))]
    titles = [f"{cat_of(t['path'])}", "recon (ceiling)",
              f"resid {rF:.3f} | " + "+".join(cat_of(pool_paths[j])[:6] for j in aF),
              f"resid {rK:.3f} | " + "+".join(cat_of(pool_paths[j])[:6] for j in aK)]
    for c in range(4):
        axes[r, c].imshow(imgs[c]); axes[r, c].axis("off")
        axes[r, c].set_title(titles[c], fontsize=7.5)
plt.suptitle("Held-out targets coded as 3-atom barycentric mixes of the basis", y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p37_basis_reconstructions.png"), dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Cell P38: decode-path control — self-coding through the pipeline
# Codes each target with ITS OWN cloud as the only atom (w=1). Output should
# match the CNT recon. Also fixes two bugs from P37:
#   (1) true (H, W) carried from the style map — no isqrt guessing
#   (2) transport-then-blend done per-point BEFORE any cross-atom mixing
# ============================================================
@torch.no_grad()
def encode_full(t):
    """Return the target's style map with true dims + flat cloud in map order."""
    sm = build_style_maps(t["bf"], t["si"], FMS)[largest][0]     # (C, H, W)
    C_, H_, W_ = sm.shape
    Ff = sm.permute(1, 2, 0).reshape(-1, C_)                     # row-major H*W
    return Ff, C_, H_, W_

@torch.no_grad()
def decode_coded_v2(t, atoms, w, hard=True):
    """Fixed decode: per-atom transported fields on the FULL grid, single
    dominant-atom hard map (no cross-atom feature averaging when hard=True
    and one weight dominates); true dims from the style map."""
    Ff, C_, H_, W_ = encode_full(t)
    X = t["X"].to(Ff.device)
    nn = torch.cdist(Ff, X).argmin(dim=1)          # full grid -> support points
    w = torch.as_tensor([float(x) for x in w], device=Ff.device)
    w = w / w.sum().clamp_min(1e-12)
    if hard and float(w.max()) > 0.6:
        # dominant atom: pure hard map, zero averaging — the contraction-safe path
        j = atoms[int(w.argmax())]
        mapped = hard_map(X, POOL[j])
    else:
        mapped = None
        for wi, j in zip(w, atoms):
            if float(wi) < 1e-4: continue
            m_ = hard_map(X, POOL[j]) if hard else None
            mapped = wi * m_ if mapped is None else mapped + wi * m_
    full = mapped[nn]
    sm_t = full.reshape(H_, W_, C_).permute(2, 0, 1).unsqueeze(0)
    smaps = {s: sm_t if s == largest else F.interpolate(
        sm_t, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    return decode_with_style_maps(t["blob"], smaps)

# CONTROL: atom = the target's own cloud
fig, axes = plt.subplots(3, len(fig_targets), figsize=(2.7 * len(fig_targets), 8.4))
ctrl_scores = []
for c, t in enumerate(fig_targets):
    with torch.no_grad():
        sm0 = build_style_maps(t["bf"], t["si"], FMS)
        recon = decode_with_style_maps(t["blob"], sm0)
        POOL_SELF = t["X"]                          # temporary self-atom
        POOL.append(POOL_SELF)
        self_dec = decode_coded_v2(t, [len(POOL) - 1], [1.0])
        POOL.pop()
        d_ctrl = vgg_dist.distance(self_dec, recon)
        d_scale = vgg_dist.distance(recon, decode_interp(
            category_data[cats_all[0]]["blob_a_gen"],
            category_data[cats_all[0]]["bf_a"], category_data[cats_all[0]]["si_a"]))
    ctrl_scores.append(d_ctrl / max(d_scale, 1e-9))
    for r, (im, ttl) in enumerate([(PILImage.open(t["path"]).convert("RGB"), "original"),
                                   (tensor_to_image(recon), "recon"),
                                   (tensor_to_image(self_dec),
                                    f"self-coded  {ctrl_scores[-1]:.2f}")]):
        axes[r, c].imshow(np.asarray(im)); axes[r, c].axis("off")
        axes[r, c].set_title(ttl, fontsize=8)
plt.suptitle("P38 control: self-coding must ~equal recon", y=1.01)
plt.tight_layout(); plt.show()
print(f"self-coding VGG shift / inter-texture scale: "
      f"{np.mean(ctrl_scores):.3f} ± {np.std(ctrl_scores):.3f}")
print("GATE: ~<0.3 => decode path is sound, proceed to P39-P41.")
print("      large => remaining pipeline bug; stop and paste the figure here.")

In [ ]:
def offset_of(X):
    return float((X.mean(0).cpu() - global_mean).norm().item() / spreads.mean())

def build_target(p):
    img = load_texture(p)
    blob, bf, si = get_generator_inputs(img)
    sm = build_style_maps(bf, si, FMS)[largest][0]
    Ff = sm.permute(1, 2, 0).reshape(-1, sm.shape[0])
    X = Ff[torch.randperm(Ff.shape[0], device=Ff.device)[:SUPPORT_N]]
    return dict(path=p, blob=blob, bf=bf, si=si, X=X.cpu(),
                off=offset_of(X), mu=X.mean(0).cpu(), sc=rms_amp(X))

In [ ]:
# P39(a) fixed: bounded, verbose target drawing
eval_targets, flagged_t, fails = [], [], []
seen = set(pool_paths)
attempts = 0
while len(eval_targets) < 30 and attempts < 120:
    attempts += 1
    c = rng.choice(cats_all)
    fs = _glob.glob(os.path.join(dtd_root, c, "*.jpg"))
    if not fs: continue
    p = rng.choice(fs)
    if p in seen: continue
    seen.add(p)
    try:
        t = build_target(p)
        (flagged_t if t["off"] > 4.0 else eval_targets).append(t)
        if len(eval_targets) % 5 == 0:
            print(f"  {len(eval_targets)}/30 clean ({len(flagged_t)} flagged, "
                  f"{len(fails)} failed, {attempts} attempts)")
    except Exception as e:
        fails.append((os.path.basename(p), type(e).__name__))
print(f"\ndone: {len(eval_targets)} clean, {len(flagged_t)} flagged, {len(fails)} failed")
if fails[:5]: print("failures:", fails[:5])
assert len(eval_targets) >= 20, "too many failures — investigate the exception types above"

In [ ]:
# what state is the loop in?
print("eval_targets:", len(eval_targets) if 'eval_targets' in dir() else "undefined",
      "| flagged:", len(flagged_t) if 'flagged_t' in dir() else "undefined")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated / "
          f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")
# try ONE encode with the error visible
c = cats_all[5]
p = _glob.glob(os.path.join(dtd_root, c, "*.jpg"))[0]
import time as _t; t0 = _t.time()
t = build_target(p)          # no try/except — let it show the real error
print(f"one encode: {_t.time()-t0:.1f}s, offset={t['off']:.2f}")

In [ ]:
POOL_MU = torch.stack([P.mean(0) for P in POOL[:N_POOL]])
POOL_SC = torch.tensor([rms_amp(P) for P in POOL[:N_POOL]])

@torch.no_grad()
def norm_signature(X, mu, sc):
    return torch.sort((((X - mu.to(X.device)) / sc).to(dev) @ PROJ), dim=0).values.reshape(-1)

NSIGS = torch.stack([norm_signature(POOL[i], POOL_MU[i], POOL_SC[i])
                     for i in range(N_POOL)])
for t in eval_targets:
    t["nsig"] = norm_signature(t["X"], t["mu"], t["sc"])
print(f"NSIGS: {tuple(NSIGS.shape)}, targets signed: {len(eval_targets)}")

In [ ]:
# ============================================================
# Cell P40: metric calibration BEFORE any re-tournament
# On 10 targets x {best-fit codes from both bases}: does the new
# shape-residual correlate with VGG distance of the DECODED result?
# No tournament runs until this r is known.
# ============================================================
from itertools import combinations
@torch.no_grad()
def code_nsig(t, basis_idx):
    Bs = NSIGS[torch.tensor(basis_idx)]
    d = (Bs - t["nsig"][None]).pow(2).mean(1)
    short = torch.argsort(d)[:8]
    best = (None, None, np.inf)
    for combo in combinations(short.tolist(), K_SPARSE):
        Bm = Bs[list(combo)]
        G = Bm @ Bm.T; b = Bm @ t["nsig"]
        lr = 1.0 / max(float(torch.linalg.eigvalsh(G.double()).max()), 1e-12)
        w = torch.full((K_SPARSE,), 1 / K_SPARSE, device=Bm.device)
        for _ in range(120):
            w = simplex_project(w - lr * (G @ w - b))
        r = float((Bm.T @ w - t["nsig"]).norm() / t["nsig"].norm())
        if r < best[2]:
            best = ([int(basis_idx[i]) for i in combo], w.clone(), r)
    return best

@torch.no_grad()
def decode_coded_v3(t, atoms, w):
    """decode with re-injection of the target's own mean/scale (the explicit
    code components) after shape transport in normalized space."""
    Ff, C_, H_, W_ = encode_full(t)
    X = t["X"].to(Ff.device)
    Xn = (X - t["mu"].to(X.device)) / t["sc"]
    w_ = torch.as_tensor([float(x) for x in w], device=X.device)
    w_ = w_ / w_.sum().clamp_min(1e-12)
    mapped = None
    for wi, j in zip(w_, atoms):
        if float(wi) < 1e-4: continue
        Yn = ((POOL[j] - POOL_MU[j]) / POOL_SC[j]).to(X.device)
        m_ = hard_map(Xn, Yn)
        mapped = wi * m_ if mapped is None else mapped + wi * m_
    mapped = mapped * t["sc"] + t["mu"].to(X.device)      # mean/scale re-injection
    nn = torch.cdist(Ff, X).argmin(dim=1)
    full = mapped[nn]
    sm_t = full.reshape(H_, W_, C_).permute(2, 0, 1).unsqueeze(0)
    smaps = {s: sm_t if s == largest else F.interpolate(
        sm_t, size=s, mode="bilinear", align_corners=False) for s in set(FMS)}
    return decode_with_style_maps(t["blob"], smaps)

pairs_cal = []
for t in eval_targets[:10]:
    with torch.no_grad():
        recon = decode_with_style_maps(t["blob"], build_style_maps(t["bf"], t["si"], FMS))
    for basis in [BASIS_FPS, BASIS_KMED]:
        atoms, w, r_sig = code_nsig(t, basis)
        with torch.no_grad():
            dec = decode_coded_v3(t, atoms, w)
            d_vgg = vgg_dist.distance(dec, recon)
        pairs_cal.append((r_sig, d_vgg))
r_sig_a = np.array([p[0] for p in pairs_cal]); d_vgg_a = np.array([p[1] for p in pairs_cal])
sp = spearmanr(r_sig_a, d_vgg_a)
print(f"shape-residual vs decoded-VGG: spearman r={sp.statistic:+.3f} (p={sp.pvalue:.1e})")
print("GATE: r > ~0.5 => the metric tracks decoded quality; rerun the tournament (P41).")
print("      r ~ 0   => signature residual is decoupled from output even after the")
print("      fixes — the tournament question is unanswerable in signature space, and")
print("      the honest evaluation is decoded-VGG directly (slower, but n=30 is fine).")

In [ ]:
# ============================================================
# Cell P41: re-tournament under the calibrated protocol (run only if P40 gates)
# FPS-40 vs kmedoids-80 (the two survivors), screened targets, new code space,
# scored BOTH ways: shape-residual and decoded-VGG. Renders the hero grid
# with typical/best/worst selection built in.
# ============================================================
BASES = {"fps40": BASIS_FPS, "kmed80": BASIS_KMED}
rows41 = []
for t in eval_targets:
    with torch.no_grad():
        recon = decode_with_style_maps(t["blob"], build_style_maps(t["bf"], t["si"], FMS))
    rec = dict(path=t["path"])
    for name, basis in BASES.items():
        atoms, w, r_sig = code_nsig(t, basis)
        with torch.no_grad():
            dec = decode_coded_v3(t, atoms, w)
            d_vgg = vgg_dist.distance(dec, recon)
        rec[f"rsig_{name}"] = r_sig
        rec[f"vgg_{name}"] = d_vgg
        rec[f"atoms_{name}"] = atoms; rec[f"w_{name}"] = w
    rows41.append(rec)

for m in ["rsig", "vgg"]:
    a = np.array([r[f"{m}_fps40"] for r in rows41])
    b = np.array([r[f"{m}_kmed80"] for r in rows41])
    w_ = wilcoxon(a, b)
    print(f"{m:5s}: fps40 {np.median(a):.4f}  kmed80 {np.median(b):.4f}  "
          f"wilcoxon p={w_.pvalue:.3f}")

# hero grid: 2 typical (median vgg) + best + worst, stated as such
order = np.argsort([r["vgg_fps40"] for r in rows41])
picks = [order[len(order)//2], order[len(order)//2 + 1], order[0], order[-1]]
tags = ["typical", "typical", "best", "worst"]
fig, axes = plt.subplots(len(picks), 4, figsize=(12.5, 3.1 * len(picks)))
for r_i, (pi, tag) in enumerate(zip(picks, tags)):
    t = eval_targets[pi]; rec = rows41[pi]
    with torch.no_grad():
        recon = decode_with_style_maps(t["blob"], build_style_maps(t["bf"], t["si"], FMS))
        dF = decode_coded_v3(t, rec["atoms_fps40"], rec["w_fps40"])
        dK = decode_coded_v3(t, rec["atoms_kmed80"], rec["w_kmed80"])
    ims = [np.asarray(PILImage.open(t["path"]).convert("RGB")),
           tensor_to_image(recon), tensor_to_image(dF), tensor_to_image(dK)]
    ttls = [f"[{tag}] {cat_of(t['path'])}", "recon (ceiling)",
            f"FPS-40 vgg {rec['vgg_fps40']:.1f} | " +
            "+".join(cat_of(pool_paths[j])[:6] for j in rec["atoms_fps40"]),
            f"kmed-80 vgg {rec['vgg_kmed80']:.1f} | " +
            "+".join(cat_of(pool_paths[j])[:6] for j in rec["atoms_kmed80"])]
    for c in range(4):
        axes[r_i, c].imshow(ims[c]); axes[r_i, c].axis("off")
        axes[r_i, c].set_title(ttls[c], fontsize=7.5)
plt.suptitle("Calibrated coding: typical / best / worst (selected by rank, not by eye)", y=1.003)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "p41_calibrated_reconstructions.png"), dpi=150,
            bbox_inches="tight")
plt.show()